# AI Agent Week 2: Transformer Cont.

04-11-2026

HF Transformers Course Implementation

## Last Week's Recap
* LLM Model structure: self attention (Q, K, V), multi-head, positional encoding,  (MoE) Feed-Forward Network (FFN). 
* Three architecture families: Encoder (BERT), Decoder (GPT), Encoder-Decoder (T5) and why they suits for different tasks.
* LLM model alone suitable for text/coding generation. But need Tools, external knowledge, and memory to be more useful in real world applications.
* Minimal agent loop. 

# This Week
 
**HF Transformers Course Ch1-3.**

**HF Pipeline Tasks**
- Tasks v.s. Models
- Pre-trained v.s. Fine-tuned models

**Tokenization**
- Padding and truncation
- Special tokens
- Mask: Causal, Padding. 
- Embeddings

**HF Trainer API**
- Mixed precision
- Training arguments
- Learning Curve

**LLM Continuation**
**HF Accelerate**
- Distributed training

**BERT Model Overview**

**KV Cache**
 

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

In [5]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification"  ,model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",                                                        
      revision="714eb0f",    )
 
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ],
    candidate_labels=["abjection", "acceptance"],
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


[{'sequence': "I've been waiting for a HuggingFace course my whole life.",
  'labels': ['acceptance', 'abjection'],
  'scores': [0.9943423867225647, 0.005657561123371124]},
 {'sequence': 'I hate this so much!',
  'labels': ['acceptance', 'abjection'],
  'scores': [0.8495982885360718, 0.15040171146392822]}]

* you need different models for different tasks. Models are fine-tuned for specific tasks and can't be swapped  interchangeably

In [8]:
generator = pipeline("text-generation", model="openai-community/gpt2", revision="607a30d")
generator("In order to be healthy, you should", max_new_tokens=20, num_return_sequences=1)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In order to be healthy, you should be able to get the most from your diet.\n\nAs a result, dieting is very'}]

In [11]:
generator = pipeline("text-generation", model="distilgpt2")
generator("In order to be healthy, you should", max_new_tokens=20, num_return_sequences=1) 

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In order to be healthy, you should have a diet that is based on a physical activity, not a lifestyle or diet that includes a diet'}]

In [12]:
unmasker = pipeline("fill-mask")
unmasker("To keep healthy, you must eat <mask> food.", top_k=2)

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'score': 0.3501476049423218,
  'token': 2245,
  'token_str': ' healthy',
  'sequence': 'To keep healthy, you must eat healthy food.'},
 {'score': 0.3332260251045227,
  'token': 30426,
  'token_str': ' nutritious',
  'sequence': 'To keep healthy, you must eat nutritious food.'}]

In [15]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'score': 0.6949763894081116, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

*an example of LLM model limitation and how to fix it*

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
question_answerer(
    question="What type of chocolate do I like?", 
    context="I do not like to eat sugar or milk",
)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.5453944206237793,
 'start': 21,
 'end': 34,
 'answer': 'sugar or milk'}

**the result is not working**

BERT has no reasoning ability. It cannot:                                                                                                                                                                          
  - Infer what you do like from what you don't like                                                                                                                                                                
  - Generate words not present in the context                                                                                                                                                                      
  - Understand negation ("do not like") logically

In [39]:

qa = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")                                                                                                                                     
qa("Question: What type of chocolate do I like? Context: I do not like sugar or milk.", max_new_tokens=20)  

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Question: What type of chocolate do I like? Context: I do not like sugar or milk. I like dark chocolate.\n\nExample: He likes a chocolate with a'}]


**only generative model have the reasoning ability. BERT can only get information from context.**

## One Model for All Tasks?

No — you need different models for different tasks. Models are fine-tuned for specific tasks and can't be swapped interchangeably.

| Task | Example Model |
|---|---|
| `text-generation` | `openai-community/gpt2` |
| `sentiment-analysis` | `distilbert/distilbert-base-uncased-finetuned-sst-2-english` |
| `zero-shot-classification` | `facebook/bart-large-mnli` |
| `fill-mask` | `distilbert/distilbert-base-uncased` |
| `summarization` | `sshleifer/distilbart-cnn-12-6` |
| `question-answering` | `distilbert/distilbert-base-cased-distilled-squad` |
| `ner` | `dbmdz/bert-large-cased-finetuned-conll03-english` |

**Why:** A GPT-2 (decoder-only, trained for next-token prediction) has no classification head — it literally cannot output a class label. A BERT classifier (encoder-only, fine-tuned for sentiment) has no generative capability.

The closest thing to "one model for many tasks" is a **large instruction-tuned model** like `meta-llama/Llama-3.2-3B-Instruct`, but you'd use it via `text-generation` with prompting, not via task-specific pipelines.

### Architecture vs Pipeline Task

```
┌─────────────────┬─────────────────────────┬──────────────────┐
│  Architecture   │      Pipeline task      │     Example      │
├─────────────────┼─────────────────────────┼──────────────────┤
│ Encoder-decoder │ Direct API (generate()) │ flan-t5, bart    │
├─────────────────┼─────────────────────────┼──────────────────┤
│ Decoder-only    │ text-generation         │ gpt2, llama      │
├─────────────────┼─────────────────────────┼──────────────────┤
│ Encoder-only    │ question-answering      │ distilbert-squad │
└─────────────────┴─────────────────────────┴──────────────────┘
```

### Pretraining Objectives

**Masked Language Modeling (MLM):** Used by encoder models like BERT, this approach randomly masks some tokens in the input and trains the model to predict the original tokens based on the surrounding context. This allows the model to learn bidirectional context (looking at words both before and after the masked word).

**Causal Language Modeling (CLM):** Used by decoder models like GPT, this approach predicts the next token based on all previous tokens in the sequence. The model can only use context from the left (previous tokens) to predict the next token.

Encoder:
Input → Self-Attention → FFN → Output (memory)

Decoder:
1. Masked self-attention (look at past outputs)
2. Cross-attention (look at encoder output)
3. FFN (process combined info)

| Location    | Attention Type        | Q comes from  | K, V come from |
| ----------- | --------------------- | ------------- | -------------- |
| Encoder     | Self-attention        | Encoder input | Encoder input  |
| Decoder (1) | Masked self-attention | Decoder input | Decoder input  |
| Decoder (2) | Cross-attention       | Decoder       | Encoder        |



Here is a table showing the essnential difference between a encoder and decoder model:

| Feature | Encoder | Decoder |
| --- | --- | --- |
| Attention type | Bidirectional (full) self-attention | Causal (masked) self-attention |
| Each token sees | All other tokens (left + right) | Only past tokens (left only) |
| Mask applied? | No | Yes — future tokens are masked out |
| Purpose | Build rich contextual representations | Generate next token autoregressively |


![The architecture of encoder-decoder models](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_architecture.png)

Such structure is originally desifned for translation, but it can be used for any task that requires understanding the relationship between two sequences of text. The encoder processes the input sequence and creates a representation of it, while the decoder generates the output sequence based on that representation.
```
Sequence: A B C D

Input:    A B C
Model:    predicts next tokens
Output:   B C D
```
```
causal mask: Decoder Self-Attention is masked to prevent attending to future tokens, ensuring the model can only use information from previous tokens when generating the next token.
padding mask: Cross Attention is masked to ignore padding tokens in the encoder output, ensuring the model focuses only on meaningful input tokens when generating output.
```


| Component                     | Padding (same length) | Padding mask (ignore PAD) | Causal mask |
| ----------------------------- | --------------------- | ------------------------- | ----------- |
| Encoder input                 | ✅ Yes                 | —                         | —           |
| Encoder self-attention        | —                     | ✅ Yes                     | ❌ No        |
| Decoder input                 | ✅ Yes                 | —                         | —           |
| Decoder masked self-attention | —                     | ✅ Yes                     | ✅ Yes       |
| Decoder cross-attention       | —                     | ✅ Yes (on encoder side)   | ❌ No        |



### Decoder-only models (e.g., GPT-2)
 designed to generate text based on a given input. They use masked self-attention, which allows them to attend only to previous tokens in the sequence, making them suitable for tasks like text generation and language modeling. However, they may struggle with tasks that require understanding the relationship between two sequences of text, such as question-answering or translation.
 other decoder-only models include LLaMA, Falcon, Mistral, etc. They are all trained with causal language modeling (CLM) objective and use masked self-attention. They can be used for a wide range of tasks via prompting, but they may not perform as well as encoder-decoder models on tasks that require understanding the relationship between two sequences of text.

![Decoder Only Model GPT-2](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/gpt2_architecture.png)

* Encoder - Decoder model family

Original Transformer — proved attention alone is sufficient for seq2seq; trained for translation
T5 — same architecture, pretrained at scale, everything is text-to-text
BART — same architecture, pretrained by learning to reconstruct corrupted text, especially strong at generation tasks like summarization

# 2. Tokenizer

* Tokenizer: Converts text into tokens and then into IDs that the model can process.
    Tensors only accept rectangular shapes
* Padding: Adding special tokens to ensure all input sequences are the same length for batch processing.
* Truncation: Shortening input sequences that exceed the model's maximum length by removing tokens from the end (or beginning) of the sequence.
* Attention Mask: A binary mask that indicates which tokens should be attended to (1) and which should be ignored (0) during model processing.

All the three, padding, truncation and attention mask, are necessary for batch processing of sequences of varying lengths. They ensure that the model can handle inputs of different sizes without running into issues with tensor shapes or attending to irrelevant tokens.


Translating text to numbers is known as encoding. Encoding is done in a two-step process: the tokenization, followed by the conversion to input IDs.

In [10]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [20]:
text_to_tokenize = "Jim Henson was a prompt engineer"

In [21]:
tokenized_text = text_to_tokenize.split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'prompt', 'engineer']


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer(text_to_tokenize)

{'input_ids': [101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [54]:

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer(sequences)

{'input_ids': [[101, 146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 102], [101, 1573, 1138, 146, 106, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [23]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
tokenizer(text_to_tokenize)

{'input_ids': [101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
Note: AutoTokenizer is just a factory/router. When you call AutoTokenizer.from_pretrained("bert-base-cased"), it reads the model config, sees the model type is bert, and internally instantiates a BertTokenizer. They
   end up being the same object class.
                                                                                                                                                                                                                   
  

In [24]:
from transformers import AutoTokenizer, BertTokenizer

t1 = AutoTokenizer.from_pretrained("bert-base-cased")
t2 = BertTokenizer.from_pretrained("bert-base-cased")
                                                                                                                                                                                                                
type(t1)  # → <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>                                                                                                                                 , type(t2)  # → <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>      

transformers.models.bert.tokenization_bert.BertTokenizer

In [25]:
type(t2) 

transformers.models.bert.tokenization_bert.BertTokenizer

In [33]:
from transformers import pipeline

qa = pipeline("text-generation", model="google/flan-t5-base")     
qa("Question: What type of chocolate do I like? Context: I do not like to eat sugar or milk.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

[{'generated_text': 'Question: What type of chocolate do I like? Context: I do not like to eat sugar or milk.e which make of eating sweetness too important by making some edible item like in sweet food: food made without additivening any parts/to ensure proper digestion This article or that chocolate you order, including raw sweet sugar which comes naturally to humans due its acid metabolism should be your health goal at work so that it might come after other nutrients you needed the food being passed as input during preparation because even those living or making such use tend some'}]

In [35]:
decoded_string = tokenizer.decode([101, 3104, 1124, 15703, 1108, 170, 5250, 18378, 3806, 102])
print(decoded_string)

[CLS] Jim Henson was a prompt engineer [SEP]


In [40]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [ ]:
### What is padding? 

In [72]:
 
batched_ids = [
    [100, 200, 200],
    [100, 200, tokenizer.pad_token_id],
]

 
print(model(torch.tensor(batched_ids)).logits)

tensor([[ 0.6514, -0.5004],
        [ 0.6465, -0.5023]], grad_fn=<AddmmBackward0>)


In [73]:
print(model(torch.tensor([batched_ids[0]])).logits)
print(model(torch.tensor([batched_ids[1]])).logits)

tensor([[ 0.6514, -0.5004]], grad_fn=<AddmmBackward0>)
tensor([[ 0.6465, -0.5023]], grad_fn=<AddmmBackward0>)


In [74]:
 
attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 0.6514, -0.5004],
        [ 0.5223, -0.3537]], grad_fn=<AddmmBackward0>)


In [70]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[100, 200, 200]]
sequence2_ids = [[100, 200]]
batched_ids = [
    [100, 200, 200],
    [100, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 0.6514, -0.5004]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5223, -0.3537]], grad_fn=<AddmmBackward0>)
tensor([[ 0.6514, -0.5004],
        [ 0.6465, -0.5023]], grad_fn=<AddmmBackward0>)


In [42]:

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [46]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(text_to_tokenize, padding="longest")
print(f"print {model_inputs}")
# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(text_to_tokenize, padding="max_length")
print(f"print {model_inputs}")

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(text_to_tokenize, padding="max_length", max_length=8)
print(f"print {model_inputs}")


print {'input_ids': [101, 3958, 27227, 2001, 1037, 25732, 3992, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
print {'input_ids': [101, 3958, 27227, 2001, 1037, 25732, 3992, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
 
# Will truncate the sequences that are longer than the model max length
# (512 for BERT or DistilBERT)
sequences = [                                                                                                                                                                                                    
      ("I've been waiting for a HuggingFace course my whole life.", "So have I!"),                                                                                                                                 
      ("What is your name?", "My name is Sylvain."),                                                                                                                                                               
  ]  
model_inputs = tokenizer(sequences, truncation=True)
print(f"model_inputs: {model_inputs}")

# Will truncate the sequences that are longer than the specified max length
model_inputs = tokenizer(sequences, max_length=8, truncation=True)
print(f"model_inputs: {model_inputs}")
""" 

 Key points:
  - [CLS] always gets 0 (belongs to sentence A)                                                                                                                                                                    
  - Both [SEP] tokens: the first gets 0 (end of A), the second gets 1 (end of B) — wait, actually both SEPs follow their respective sentence's id
  - The switch from 0 → 1 happens right after the first [SEP]    
"""      

model_inputs: {'input_ids': [[101, 146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 102, 1573, 1138, 146, 106, 102], [101, 1327, 1110, 1240, 1271, 136, 102, 1422, 1271, 1110, 156, 7777, 2497, 1394, 119, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}
model_inputs: {'input_ids': [[101, 146, 112, 1396, 102, 1573, 1138, 102], [101, 1327, 1110, 102, 1422, 1271, 1110, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 1, 1, 1], [0, 0, 0, 0, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1]]}


In [48]:
tokenizer.pad_token_id

0

In [53]:
tokenizer.cls_token_id

101

In [50]:
tokenizer.decode(tokenizer.pad_token_id)

'[PAD]'

In [52]:
tokenizer.decode(tokenizer.cls_token_id)

'[CLS]'

In [ ]:
### what is token_type_ids


In [43]:
sequence = sequence[:max_sequence_length]

NameError: name 'max_sequence_length' is not defined

In [ ]:
"""
Every Model Was Trained With One Specific TokenizerDuring training, the model learned to associate token IDs with meanings. Token ID 4732 might mean " Paris" in one tokenizer, but mean something completely different in another tokenizer's vocabulary.GPT-2 tokenizer:    "I love Paris"  →  [40, 1842, 6342]
BERT tokenizer:     "I love Paris"  →  [1045, 2293, 3000]Same words, completely different numbers. The model only ever saw one of these mappings during training.
"""
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "Living a healthy life need patience. But it is a must."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input IDs: tensor([[ 2542,  1037,  7965,  2166,  2342, 11752,  1012,  2021,  2009,  2003,
          1037,  2442,  1012]])
Logits: tensor([[-0.3184,  0.4760]], grad_fn=<AddmmBackward0>)


In [60]:
import torch.nn.functional as F   
probs = F.softmax(output.logits, dim=-1)
print(probs)    

tensor([[0.3112, 0.6888]], grad_fn=<SoftmaxBackward0>)


In [62]:
tokens

['living',
 'a',
 'healthy',
 'life',
 'need',
 'patience',
 '.',
 'but',
 'it',
 'is',
 'a',
 'must',
 '.']

In [ ]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print("Input IDs:", ids)

Input IDs: [2542, 1037, 7965, 2166, 2342, 11752, 1012, 2021, 2009, 2003, 1037, 2442, 1012]


* what is ids?
* what is tokens?
* what is the difference between tokens and ids? - Answer: Tokens are the human-readable pieces of text that the model processes, while IDs are the numerical representations of those tokens that the model actually uses for computation. The tokenizer converts text into tokens and then maps those tokens to their corresponding IDs based on the model's vocabulary.



## Embedding 
maps discrete token IDs (a sparse representation) into dense continuous vectors where semantic and syntactic relationships can be learned.

# Context Length and attention span

## Optimize inference with batching and attention masks

# How inference work? 

For an Encoder-Decoder model (e.g. T5, original Transformer)
Your first framing is more accurate here.
Prefill  →  Encoder runs over input  →  produces K, V representations
                                              ↓
Decode   →  Decoder cross-attends to those K, V  →  generates output tokens
Here prefill is closer to "produce the context that the decoder will cross-attend to." The encoder output is essentially a rich KV bank. The first output token is generated in the first decode step, not prefill.

For a Decoder-Only model (e.g. GPT, LLaMA, Claude)
Your second framing is more accurate here.
Prefill  →  All prompt tokens processed in parallel
         →  KV cache populated for all prompt positions
         →  Logits computed at final prompt position
         →  First output token sampled   ← happens HERE, at end of prefill
There is no separate encoder. The "understanding" and "first generation" happen in the same forward pass. Prefill ends the moment token 1 is produced.


- Prefill
Based on inputs, go
During prefill, the model processes all input tokens in parallel and produces:

The KV cache — computed K and V matrices for every input token, stored for future decode steps
The logits for the last token — which means the first output token is generated at the end of prefill

- Decode

In decoder model, generate the first token is part of the prefill. In encoder-decoder model, the first token is generated at the end of prefill. After that, the model generates one token at a time, using the KV cache to attend to all previous tokens without recomputing attention for them. 

Keep the result fresh
Presence Penalty: A fixed penalty applied to any token that has appeared before, regardless of how often. This helps prevent the model from reusing the same words.
Frequency Penalty: A scaling penalty that increases based on how often a token has been used. The more a word appears, the less likely it is to be chosen again.

Beam Search: A decoding strategy that keeps track of multiple candidate sequences (beams) at each step, allowing the model to explore different possibilities and select the most likely sequence as output. This can lead to more coherent and contextually relevant responses compared to greedy decoding, which only considers the most probable token at each step.


## What is KV cache?

The KV cache (Key-Value cache) is a mechanism used in transformer models to store the computed Key (K) and Value (V) matrices for each token during the prefill phase. This allows the model to efficiently attend to previously processed tokens during the decoding phase without having to recompute the attention for those tokens, thus speeding up inference.


Prefill is fast because all tokens are processed in parallel — it's big matrix multiplications which GPUs love. Decode is slow because it's sequential and bottlenecked by memory bandwidth of loading the growing KV cache on every single step.
This is why in LLM serving, a long prompt (heavy prefill) costs less time than generating a long output (many decode steps) — and why techniques like speculative decoding and KV cache compression target the decode phase specifically.


 
Two parts matters

**GPU**
├── Compute  →  the math-doing units (tensor cores, CUDA cores)
│                  "how fast can I multiply matrices?"
└── Memory   →  where data lives while working
    ├── VRAM (HBM)   →  large but slower  (~40-80GB on A100)
    └── SRAM (cache) →  tiny but fast     (~40MB on A100)

These two are separate resources with a wire between them. The wire has limited bandwidth — this is the root cause of almost every optimization technique on that page.

**WHAT IT FIXES**

- **Flash Attention**  →   Compute efficiency
  (reduce redundant VRAM↔SRAM transfers during attention)
         →  fixes HOW attention is COMPUTED             
- **PagedAttention**   →   Memory efficiency  
  (eliminate wasted KV cache space, reduce fragmentation)
KV Cache = the thing being stored/accessed
 →  fixes HOW KV cache is STORED in memory

- **Quantization**     →   Memory capacity
Model weights are stored as floating point numbers. By default, fp16 = 2 bytes per parameter.
Quantization     →  fixes HOW BIG everything is, including KV cache

  (shrink model so it fits in VRAM at all)
 

# Fine Tune

https://huggingface.co/docs/transformers/tasks/language_modeling#causal-language-modeling 

# Bert 

BERT is an encoder-only model and is the first model to effectively implement deep bidirectionality to learn richer representations of the text by attending to words on both sides.

https://huggingface.co/docs/transformers/tasks/sequence_classification

# Inference

In [ ]:
YOUR_HF_TOKEN

In [5]:
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct")
result = pipe("Tell me a story")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


max_new_tokens=100                                                                                                                                                                                               
  Maximum number of tokens to generate. Generation stops at 100 new tokens even if no stop condition is hit.                                                                                                       
                                                                                                                                                                                                                   
  ---                                                                                                                                                                                                              
  temperature=0.7                                                                                                                                                                                                  
  Controls randomness by scaling logits before sampling.                                                                                                                                                           
                  
  ┌───────┬────────────────────────────────────────────────────────┐                                                                                                                                               
  │ Value │                         Effect                         │
  ├───────┼────────────────────────────────────────────────────────┤                                                                                                                                               
  │ 0.0   │ Deterministic — always picks highest probability token │
  ├───────┼────────────────────────────────────────────────────────┤
  │ 0.7   │ Moderately creative, still coherent                    │                                                                                                                                               
  ├───────┼────────────────────────────────────────────────────────┤
  │ 1.0   │ Sampling at true model distribution                    │                                                                                                                                               
  ├───────┼────────────────────────────────────────────────────────┤                                                                                                                                               
  │ >1.0  │ More random, less coherent                             │
  └───────┴────────────────────────────────────────────────────────┘                                                                                                                                               
                  
  ---                                                                                                                                                                                                              
  top_p=0.95      
  Nucleus sampling — at each step, only consider tokens whose cumulative probability adds up to 95%, then sample from that subset.
                                                                                                                                  
  Why 0.95 specifically? It's a widely used default that:                                                                                                                                                          
  - Cuts off the long tail of unlikely tokens (the bottom 5%)                                                                                                                                                      
  - Keeps enough candidates for variety without introducing incoherence                                                                                                                                            
                                                                                                                                                                                                                   
  ┌───────┬─────────────────────────────────────────────────┐                                                                                                                                                      
  │ Value │                     Effect                      │
  ├───────┼─────────────────────────────────────────────────┤                                                                                                                                                      
  │ 1.0   │ Consider all tokens (no filtering)              │
  ├───────┼─────────────────────────────────────────────────┤
  │ 0.95  │ Use top tokens covering 95% of probability mass │                                                                                                                                                      
  ├───────┼─────────────────────────────────────────────────┤                                                                                                                                                      
  │ 0.5   │ Very conservative, only high-confidence tokens  │                                                                                                                                                      
  └───────┴─────────────────────────────────────────────────┘                                                                                                                                                      
                                                                                                                                                                                                                 
  temperature and top_p are typically used together — temperature reshapes the distribution, top_p then filters it.                                                                                                
  
  ---                                                                                                                                                                                                              
  details=True                                                                                                                                                                                                   
  Returns extra metadata alongside the generated text — token-level log probabilities, finish reason (length, eos_token, stop_sequence), and per-token details. Useful for debugging or analysis.

  ---                                                                                                                                                                                                              
  stop_sequences=[]
  List of strings that immediately halt generation if produced. Empty list means no custom stop conditions — generation runs until max_new_tokens or the model's EOS token.     



  Data Collator
Data collators are objects that form batches by using a list of dataset elements as input. They may apply processing like padding to create consistent batch sizes and can also apply data augmentation techniques.    

# Fine Tune

* in this example, we will show how to use a dataset to fine tune a bert model.

The dataset is from GLUE, and the task is called "MRPC". The dataset contains pairs of sentences and a label indicating whether the second sentence is a paraphrase of the first sentence.

In [28]:
!pip install datasets evaluate transformers[sentencepiece]

In [30]:
import numpy as np
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification 
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset


In [1]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
YOUR_HF_TOKEN

In [12]:

# Same as before
checkpoint = "bert-base-uncased"
# model and tokenizer should come from the same checkpoint
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:

sequences = [
    "I've been waiting for a HuggingFace course my whole life. And it is what I want",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
print(batch) 
batch["labels"] = torch.tensor([1, 1])
print(batch.labels) 


{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,  1998,  2009,  2003,  2054,  1045,
          2215,   102],
        [  101,  2023,  2607,  2003,  6429,   999,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}
tensor([1, 1])


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# UNEXPECTED — weights that exist in the checkpoint (bert-base-uncased) but are not used by BertForSequenceClassification:                                                           
#  - These are the MLM (Masked Language Modeling) and NSP (Next Sentence Prediction) heads that BERT was originally pretrained with — they're simply discarded.
# swapping BERT's pretraining heads for a new classification head. This is expected and intentional — it's the whole point of fine-tuning. 
# The missing weights will be learned during your training loop.     


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [45]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [46]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [47]:
raw_train_dataset.features.keys()

dict_keys(['sentence1', 'sentence2', 'label', 'idx'])

In [ ]:
raw_train_dataset[10]

{'sentence1': 'Legislation making it harder for consumers to erase their debts in bankruptcy court won overwhelming House approval in March .',
 'sentence2': 'Legislation making it harder for consumers to erase their debts in bankruptcy court won speedy , House approval in March and was endorsed by the White House .',
 'label': 0,
 'idx': 11}

In [23]:
raw_train_dataset[15]

{'sentence1': 'Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .',
 'sentence2': 'Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .',
 'label': 0,
 'idx': 16}

In [7]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
print("Check input lengths:", [len(x) for x in samples["input_ids"]])

batch = data_collator(samples)
print("Batch shapes:", {k: v.shape for k, v in batch.items()})

print("after data collator, each sequence has been padded to the same length, and the batch is ready to be fed into the model.")

Check input lengths: [50, 59, 47, 67, 59, 50, 62, 32]
Batch shapes: {'input_ids': torch.Size([8, 67]), 'token_type_ids': torch.Size([8, 67]), 'attention_mask': torch.Size([8, 67]), 'labels': torch.Size([8])}
after data collator, each sequence has been padded to the same length, and the batch is ready to be fed into the model.


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer", eval_strategy="epoch")


In [18]:
tokenized_datasets["validation"].shape

(408, 7)

In [17]:
print("validation data shape:", tokenized_datasets["validation"].shape)


validation data shape: (408, 7)


NameError: name 'trainer' is not defined

In [ ]:

predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)
preds = np.argmax(predictions.predictions, axis=-1)

NameError: name 'preds' is not defined

In [31]:

def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments("test-trainer", eval_strategy="epoch") 
# "test-trainer" is the directory where the model predictions and checkpoints will be saved. You can choose any name you want for this directory.
# eval_strategy="epoch" means that the evaluation will be performed at the end of each epoch during training. This allows you to monitor the model's performance on the validation set after each epoch and make decisions based on those results, such as early stopping or saving the best model checkpoint.
 

In [32]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args, 
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.573634,0.813725,0.875817
2,0.340914,0.600824,0.850490,0.895726
3,0.231251,0.771883,0.850490,0.897133


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.23904371296225377, metrics={'train_runtime': 210.2045, 'train_samples_per_second': 52.349, 'train_steps_per_second': 6.551, 'total_flos': 405114969714960.0, 'train_loss': 0.23904371296225377, 'epoch': 3.0})

#### Accelerater

In [38]:
 
# ["attention_mask", "input_ids", "labels", "token_type_ids"]

from torch.utils.data import DataLoader
 

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print("training steps:", num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("check if GPU is available:", torch.cuda.is_available(),"device is", device)

model.to(device)


training steps: 1377
check if GPU is available: True device is cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [39]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [40]:
# accelerate is a library that helps you easily train and evaluate your models on different hardware setups, such as CPUs, GPUs, and TPUs, without having to write complex code to handle the distribution of data and model across devices. It abstracts away the complexities of distributed training and allows you to focus on your model and training logic.

from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

In [ ]:
from accelerate import notebook_launcher

notebook_launcher(training_function)

#### mixed precision training

Mixed precision uses fp16 for speed and memory savings where it's safe, and fp32 where numerical accuracy is critical — giving you faster training, lower memory usage, and nearly identical results compared to pure fp32.

why it is called "Mixed"?


Model weights        →  fp16  (stored small, saves memory)
Gradient computation →  fp32  (needs precision, avoids errors)
Activations          →  fp16  (fast computation)
Loss scaling         →  fp32  (critical, must be accurate)


  sign  exponent  mantissa
fp32:      1      8         23     (range: huge, precision: high)
fp16:      1      5         10     (range: small, precision: medium)
bf16:      1      8          7     (range: huge, precision: lower)
bf16 keeps the same exponent size as fp32 — so it has the same numerical range, just less decimal precision. This means:

No underflow/overflow problems (same range as fp32)
Half the memory of fp32
Fast on modern GPU tensor cores

For LLMs, the range matters more than precision — so bf16 is generally preferred over fp16 for training. fp16 is more common for inference.

The Practical Picture for LLMs
Training:
  Weights stored in fp16/bf16   ← saves memory
  Forward pass in fp16/bf16     ← fast tensor core math
  Gradients computed in fp32    ← prevents underflow
  Master weights in fp32        ← accurate weight updates
  
Inference:
  Weights loaded in fp16        ← fits more model in VRAM
  Computation in fp16           ← faster than fp32
  (no gradients needed at all)  ← simpler than training

Why the Course / llama.cpp Mentions It
In the context of that page, mixed precision (specifically fp16) is relevant because:
Default model download:  fp32  →  too big for most GPUs
fp16 model:              half the size, nearly identical output quality
int4 quantization:       4× smaller, slight quality loss
When you see model files named model.fp16.bin or Q4_K_M.gguf — those are the precision choices baked into the saved weights. fp16 is the standard "I want good quality but fit in my GPU" choice. int4 is "I want to run on a laptop."

One Line Summary

Mixed precision uses fp16 for speed and memory savings where it's safe, and fp32 where numerical accuracy is critical — giving you faster training, lower memory usage, and nearly identical results compared to pure fp32.


### Understand training loop
forward → backward → optimizer step → scheduler step → zero gradients

freeze model parameter ```requires_grad=False```
change layer behavior, disable dropout and batch normalization ```model.eval()``` vs ```model.train()```. 
during evaluation to disable gradient computation ```torch.no_grad()```


Accelerator
wrap key objects with accelerator.prepare() and use accelerator.backward() instead of loss.backward()


Learning Curve Understanding: 
The loss can improve if the model’s output gets closer to the target, even if the final prediction is still incorrect. Accuracy, however, only improves when the prediction crosses the threshold to be correct.

### Advanced Optimization Techniques


In [48]:


%pip install -qqq torch torchvision setuptools scikit-learn

# Install Hugging Face libraries
%pip install  --upgrade datasets -qqq accelerate hf-transfer transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.9 MB/s eta 0:00:00:00:0100:01


In [49]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "burtenshaw/PleIAs_common_corpus_code_classification"

# Load raw dataset
dataset = load_dataset(dataset_id)

README.md:   0%|          | 0.00/418 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/94.9M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/94.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14192 [00:00<?, ? examples/s]

In [50]:
print(len(dataset["train"]))
print(dataset["train"][0])

127723
{'text': '/*\n * Copyright (c) 2000 Kungliga Tekniska Högskolan\n * (Royal Institute of Technology, Stockholm, Sweden).\n * All rights reserved.\n *\n * Redistribution and use in source and binary forms, with or without\n * modification, are permitted provided that the following conditions\n * are met:\n *\n * 1. Redistributions of source code must retain the above copyright\n *    notice, this list of conditions and the following disclaimer.\n *\n * 2. Redistributions in binary form must reproduce the above copyright\n *    notice, this list of conditions and the following disclaimer in the\n *    documentation and/or other materials provided with the distribution.\n *\n * 3. Neither the name of the Institute nor the names of its contributors\n *    may be used to endorse or promote products derived from this software\n *    without specific prior written permission.\n *\n * THIS SOFTWARE IS PROVIDED BY THE INSTITUTE AND CONTRIBUTORS ``AS IS\'\' AND\n * ANY EXPRESS OR IMPLIED W

In [51]:
from transformers import AutoTokenizer

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

config.json: 0.00B [00:00, ?B/s]

You are using a model of type modernbert to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/127723 [00:00<?, ? examples/s]

Map:   0%|          | 0/14192 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [53]:
tokenized_dataset.shape

{'train': (127723, 3), 'test': (14192, 3)}

In [55]:
tokenized_dataset['train'].features.keys()

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [58]:
from transformers import AutoModelForSequenceClassification

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Prepare model labels - useful for inference
labels = list(set(tokenized_dataset["train"]["labels"]))
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label



In [68]:
tokenized_datasets.save_to_disk("tokenized_datasets")

Saving the dataset (0/1 shards):   0%|          | 0/3668 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1725 [00:00<?, ? examples/s]

In [60]:
  !pip install -U transformers

In [65]:
  !pip install -U huggingface_hub   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 16.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 89.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0


In [63]:
import transformers   
print(transformers.__version__)  
from transformers import AutoModelForSequenceClassification

5.0.0


In [67]:
import importlib                                                                                                                                                                   
import transformers
importlib.reload(transformers)     

<module 'transformers' from '/usr/local/lib/python3.12/dist-packages/transformers/__init__.py'>

In [68]:

model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3656/2560615369.py", line 1, in <cell line: 0>
    model = AutoModelForSequenceClassification.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 317, in from_pretrained
    if not isinstance(config, PreTrainedConfig):
                         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/configuration_auto.py", line 1384, in from_pretrained
    "using the `AutoConfig.from_pretrained(pretrained_model_name_or_path)` method."
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/configuration_auto.py", line 1092, in __

: 

: 

In [56]:
 

import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

from huggingface_hub import HfFolder
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir= "ModernBERT-code-classifier",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True, # bfloat16 training
    optim="adamw_torch_fused", # improved optimizer
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # push to hub parameters
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=HfFolder.get_token(),
    report_to="wandb"
)



StrictDataclassDefinitionError: Class 'ModernBertConfig' must be a dataclass before applying @strict.

In [ ]:
limited_dataset = tokenized_dataset["train"].select(range(100))

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=limited_dataset,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()